In [ ]:
import os
from openrouter import OpenRouter
import dotenv
import json

dotenv.load_dotenv()

apiKey = os.getenv("API_KEY")
client = OpenRouter(api_key=apiKey)

rules = ""
skill = open("Agents.md", "r")
rules = skill.read()
skill.close()

while True:
    filePath = input("File Path: ")
    prompt = f"{rules} {input("Prompt: ")}"

    response = client.chat.send(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[{ "role": "user", "content": prompt }],
    )

    response = response.choices[0].message.content
    response = json.loads(response)

    if response["action"] == "coding":
        abs_path = os.path.abspath(filePath)
        parent_dir = os.path.dirname(abs_path)
        os.makedirs(parent_dir, exist_ok=True)
        
        with open(filePath, "w") as f:
            f.write(response["code"])
            print(f"Generated code at {filePath}")
    
    if response["action"] == "python_automation":
        exec(response["code"])
        print("Task successfully automated.")